In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
import gc


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"working on {device}")

In [ ]:
!pip install huggingface_hub
from huggingface_hub import login, whoami

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # add under Colab Secrets
hf_token = os.environ["HF_TOKEN"]  # set before running, e.g. via Colab Secrets
login(token=hf_token)

In [ ]:
user = whoami()
print(f"Logged in as {user['name']}")

In [ ]:
# ---------- Choose the auxiliary model used for attention scoring ----------
# "qwen":  Qwen2.5-7B on the full AgentIF test set (707 items).
#          Run this first -- its output is the input to Downsampling_Data.
# "llama": Llama-3.1-8B-Instruct on the 128-item downsampled set.
#          Run after Downsampling_Data. (Gated model: accept Meta's license on Hugging Face first.)
AUX_CHOICE = "llama"  # "qwen" or "llama"

DATA_DIR = "/content/drive/MyDrive/TokenReductionProject/Data/"
AUX_CONFIGS = {
    "qwen": {
        "model": "Qwen/Qwen2.5-7B",
        "name": "Qwen2.5-7B",
        "input": DATA_DIR + "eval_criteria.pkl",
        "suffix": "_",            # -> ..._model_Qwen2.5-7B_.pkl (read by Downsampling_Data)
    },
    "llama": {
        "model": "meta-llama/Llama-3.1-8B-Instruct",
        "name": "Llama-3.1-8B-Instruct",
        "input": DATA_DIR + "eval_criteria_DOWNSAMPLED.pkl",
        "suffix": "_DOWNSAMPLED", # -> ..._model_Llama-3.1-8B-Instruct_DOWNSAMPLED.pkl
    },
}
CFG = AUX_CONFIGS[AUX_CHOICE]

# ---------- 0) Load aux model once ----------
AUX_MODEL = CFG["model"]

_aux_tok = AutoTokenizer.from_pretrained(AUX_MODEL, use_fast=True)
if _aux_tok.pad_token is None:
    _aux_tok.pad_token = _aux_tok.eos_token

_aux_model = AutoModelForCausalLM.from_pretrained(
    AUX_MODEL,
    dtype=torch.bfloat16,
    device_map="auto" if torch.cuda.is_available() else None,
    attn_implementation="eager"
).eval()

_aux_model.config.use_cache = False

In [ ]:
AGENTIF_FILE_PATH = CFG["input"]  # full set for Qwen, downsampled set for Llama

In [ ]:
def _sent_spans(text: str):
    # returns [(char_start, char_end, span_text), ...]
    spans = []
    for m in re.finditer(r".+?(?:[.!?](?:\s+|$)|\n{2,}|$)", text, flags=re.S):
        s = m.group(0)
        if s.strip():
            spans.append((m.start(), m.end(), s))
    return spans

def _approx_tokens(s: str) -> int:
    return max(1, len(s) // 4)  # cheap heuristic; good enough for budgeting

@torch.no_grad()
def get_attention_scores(context, objective, *, top_layers=8, max_seq_len=2048, ctx_stride=256):
    context = context.strip()
    objective = objective.strip()
    if not context:
        return []

    # 1) tokenize objective once
    enc_obj = _aux_tok(objective, add_special_tokens=False, return_tensors="pt")
    obj_ids = enc_obj["input_ids"][0]

    # 2) tokenize context once (with offsets)
    enc_ctx = _aux_tok(context, add_special_tokens=False, return_tensors="pt",
                       return_offsets_mapping=True)
    ctx_ids = enc_ctx["input_ids"][0]
    ctx_offsets = enc_ctx["offset_mapping"][0].tolist()

    # 3) sep + chunk size
    sep_ids = _aux_tok("\n\n", add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    sep_len = sep_ids.numel()
    max_ctx_chunk = max(32, max_seq_len - (obj_ids.numel() + sep_len + 2))

    device = _aux_model.device

    # 4) CPU accumulator
    imp_cpu = torch.zeros((ctx_ids.numel(),), dtype=torch.float32)

    start = 0
    while start < ctx_ids.numel():
        end = min(ctx_ids.numel(), start + max_ctx_chunk)
        chunk_ids = ctx_ids[start:end]

        # 5a) build input: chunk + sep + objective
        input_ids = torch.cat([chunk_ids, sep_ids, obj_ids], dim=0).unsqueeze(0).to(device)
        attn_mask = torch.ones_like(input_ids)

        ctx_len_with_sep = chunk_ids.numel() + sep_len
        seq_len = input_ids.shape[1]
        q_pos = list(range(ctx_len_with_sep, seq_len))  # objective token positions

        # 5b) forward (big memory happens here, but seq_len is capped)
        out = _aux_model(
            input_ids=input_ids,
            attention_mask=attn_mask,
            output_attentions=True,
            use_cache=False,
            return_dict=False,
        )
        # depending on model, attentions are out[-1] or out[2]; safest:
        attns = out[-1]

        # 5c) compute importance for this chunk (on GPU, small)
        L = len(attns)
        l0 = max(0, L - top_layers)
        imp_chunk = torch.zeros((ctx_len_with_sep,), device=device)

        for l in range(l0, L):
            a = attns[l][0]  # (heads, seq, seq)
            imp_chunk += a[:, q_pos, :ctx_len_with_sep].sum(dim=(0, 1))

        # 5d) drop sep tokens, move to CPU, add into global accumulator
        imp_ctx_only_cpu = imp_chunk[:chunk_ids.numel()].float().cpu()
        imp_cpu[start:end] += imp_ctx_only_cpu

        # 5e) cleanup to avoid accumulation across chunks
        del out, attns, imp_chunk, imp_ctx_only_cpu, input_ids, attn_mask, a
        torch.cuda.empty_cache()
        gc.collect()

        if end == ctx_ids.numel():
            break
        start = end - ctx_stride  # overlap

    # 6) normalize once
    imp_cpu = imp_cpu / (imp_cpu.sum() + 1e-9)

    # 7) token scores -> sentence/span scores
    spans = _sent_spans(context)  # returns (char_s, char_e, text)
    # map token scores to spans via offsets overlap
    scored = []
    for i, (sp_s, sp_e, sp_txt) in enumerate(spans):
        s = 0.0
        for tok_i, (ts, te) in enumerate(ctx_offsets):
            if te <= sp_s:
                continue
            if ts >= sp_e:
                break
            if te > ts and ts < sp_e and te > sp_s:
                s += float(imp_cpu[tok_i].item())
        scored.append((i, s, sp_txt))

    return scored


In [ ]:
p = Path(AGENTIF_FILE_PATH)

def load_agentif_rows(path: Path):
    if path.suffix == ".pkl":
        obj = pd.read_pickle(path)  # could be DataFrame or list[dict]
        if isinstance(obj, pd.DataFrame):
            return obj.to_dict("records")
        return obj
    elif path.suffix == ".json":
        return json.loads(path.read_text())
    elif path.suffix in [".jsonl", ".jl"]:
        return [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

rows = load_agentif_rows(p)

In [ ]:
PROBE = "Answer: "

for row in tqdm(rows):
  context = ""
  for query in row["input"]:
      context += str(query.get("content")) + "\n"
  row["scored_spans"] = get_attention_scores(context, PROBE) #generic probe as the objective --> gets what's broadly useful in the prompt


In [ ]:
print("Loaded rows:", len(rows))
print("Example rows:", rows[1])

In [ ]:
attn_df = pd.DataFrame(rows)
AUX_MODEL_NAME = CFG["name"]
attn_df.to_pickle('/content/drive/MyDrive/TokenReductionProject/Data/eval_criteria_with_attention_scores_probe_' + PROBE +'_model_' + AUX_MODEL_NAME +CFG["suffix"] + '.pkl')

In [ ]:
attn_df.to_csv('/content/drive/MyDrive/TokenReductionProject/Data/eval_criteria_with_attention_scores_probe_' + PROBE +'_model_' + AUX_MODEL_NAME +CFG["suffix"] + '.csv')